In [0]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F
import pandas as pd
import os

In [0]:
df_comex_mun = spark.read.table("obsref.comex.flat_municipio")
df_comex_ncm = spark.read.table("obsref.comex.flat_ncm")

#Dimensão dos países
df_countries = pd.read_csv("/Workspace/Observatorio_dev/painel_comex/public/data/countries_br.csv", encoding='latin', sep=";")

# Dimensão dos municípios -> VPs
df_munic = spark.read.table("obsref.observatorio.dim_municipio")

# Dimensão dos NCMs -> Produtos -> SC Competitiva
df_dic = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQ5AE-EGJxoKed9vxCBEeFAf6LM6GRdNvWhY4npQLkHDk9muuRM6f-Tn4lcdHtd5R9CsA4TlWxn5XDu/pub?gid=1951449408&single=true&output=csv")

In [0]:
output = "/Workspace/Observatorio_dev/painel_comex/public/data"
os.makedirs(output, exist_ok=True)

In [0]:
# ------------------------- Tratando dados da base estadual (NCM)
df_comex_ncm = df_comex_ncm.filter(
    (col("nr_ano").cast("int") >= 2020) &
    (col("sg_uf") == "SC")
)

df_comex_ncm = df_comex_ncm.select(
    "nr_ano", "nr_mes", "tp_carga", "cd_sh4", "cd_pais", "ds_pais", "vl_fob"
)

df_countries = spark.createDataFrame(df_countries).select(
    F.col("CO_PAIS").alias("cd_pais"),
    F.col("CO_PAIS_ISOA3").alias("cd_pais_iso3"),
    F.col("NO_PAIS").alias("nm_pais")
)

df_comex_ncm = df_comex_ncm.join(
    F.broadcast(df_countries),
    on="cd_pais",
    how="left"
)

df_dic = spark.createDataFrame(df_dic)

df_dic_final = df_dic.select(
    "cd_sh4", "nm_produto", "nm_sc_competitiva", "ds_cnae_divisao", "ds_cnae_grupo").groupBy("cd_sh4").agg(
    F.first("nm_produto").alias("nm_produto"),
    F.first("nm_sc_competitiva").alias("nm_sc_competitiva"),
    F.first("ds_cnae_divisao").alias("ds_cnae_divisao"),
    F.first("ds_cnae_grupo").alias("ds_cnae_grupo")
)

df_comex_ncm = df_comex_ncm.join(
    F.broadcast(df_dic_final), 
    on="cd_sh4", 
    how="left"
)

In [0]:
# ------------------------- Tratando dados da base municipal (municipio)
df_comex_mun = df_comex_mun.filter(
    (col("nr_ano").cast("int") >= 2020) &
    (col("sg_uf") == "SC")
)

df_comex_mun = df_comex_mun.select([
    "nr_ano", "nr_mes", "tp_carga", "cd_sh4", "cd_pais", "cd_municipio", "nm_municipio", "vl_fob"
])

df_munic_final = df_munic.select(
    "cd_municipio", "nm_vice_presidencia").groupBy("cd_municipio").agg(
    F.first("nm_vice_presidencia").alias("nm_vice_presidencia")
)

df_comex_mun = df_comex_mun.join(
    F.broadcast(df_countries),
    on="cd_pais",
    how="left"
)

df_comex_mun = df_comex_mun.join(
    F.broadcast(df_munic_final), 
    on="cd_municipio", 
    how="left"
)

df_comex_mun = df_comex_mun.join(
    F.broadcast(df_dic_final), 
    on="cd_sh4", 
    how="left"
)

In [0]:
# ------------------------- Exportando dataframes
data_dict = df_comex_ncm.collect()
pdf_ncm = pd.DataFrame([row.asDict() for row in data_dict])

pdf_ncm.to_parquet(os.path.join(output, "painel_ncm.parquet"))

data_dict = df_comex_mun.collect()
pdf_mun = pd.DataFrame([row.asDict() for row in data_dict])

pdf_mun.to_parquet(os.path.join(output, "painel_mun.parquet"))